In [1]:
import numpy as np
import pandas as pd
import fastf1 as f1

In [2]:
circuits = pd.read_csv('data/circuits.csv')
constructor_results = pd.read_csv('data/constructor_results.csv')
constructor_standings = pd.read_csv('data/constructor_standings.csv')
constructors = pd.read_csv('data/constructors.csv')
driver_standings = pd.read_csv('data/driver_standings.csv')
drivers = pd.read_csv('data/drivers.csv')
lap_times = pd.read_csv('data/lap_times.csv')
pit_stops = pd.read_csv('data/pit_stops.csv')
qualifying = pd.read_csv('data/qualifying.csv')
races = pd.read_csv('data/races.csv')
results = pd.read_csv('data/results.csv')
sprint_results = pd.read_csv('data/sprint_results.csv')
status = pd.read_csv('data/status.csv')
hungary = f1.get_session(2024, 13, "Race")
belgium = f1.get_session(2024, 14, "Race")

req         WARNING 	DEFAULT CACHE ENABLED! (291.12 MB) /Users/jestonlewis/Library/Caches/fastf1


In [3]:
hungary.load()
belgium.load()

core           INFO 	Loading data for Hungarian Grand Prix - Race [v3.4.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['81', '4', '44', '16', '1', '55', '11', '63', '22', '18', '14', '3', '27', '23', '20', '77', '2', '31', '24', '10']
core           INFO 	Loading data for Belgian Grand Prix - R

In [4]:
results = results.drop(results[results.raceId < 989].index)
races = races.drop(races[races.year < 2018].index)

In [5]:
final = pd.merge(results, races, on="raceId")
final = pd.merge(final, circuits, on="circuitId")
final = pd.merge(final, drivers, on="driverId")

In [6]:
final.drop(columns=["resultId", "number_x", "positionText", "positionOrder", "points", "laps", "time_x", "milliseconds", "fastestLap", "rank", "fastestLapTime", "fastestLapSpeed", "statusId", "round", "name_x", "url_x", "fp1_date", "fp1_time", "fp2_date", "fp2_time", "fp3_date", "fp3_time", "quali_date", "quali_time", "sprint_date", "sprint_time", "name_y", "location", "country", "lat", "lng", "url_y", "number_y", "code", "forename", "surname", "dob", "nationality", "url", "alt"], inplace=True)

In [7]:
final.rename(columns={"time_y":"time"}, inplace=True)

In [8]:
races = [belgium, hungary]

In [9]:
for race in races:
    race_results = race.results
    race_results.drop(columns=["DriverNumber", "BroadcastName", "Abbreviation", "TeamColor", "FirstName", "LastName", "FullName", "HeadshotUrl", "CountryCode", "ClassifiedPosition", "Q1", "Q2", "Q3", "Time", "Status"], inplace=True)
    race_results = pd.merge(race_results, drivers, left_on="DriverId", right_on="driverRef")
    race_results["year"] = race.session_info["StartDate"].date().year
    race_results["date"] = race.session_info["StartDate"].date().strftime("%Y-%m-%d")
    race_results["time"] = race.session_info["StartDate"].time().strftime("%H:%M")
    if race.session_info["Meeting"]["Circuit"]["ShortName"] == "Spa-Francorchamps":
        # TODO come up with better way to add circuit names
        race_results["circuitRef"] = "spa"
        race_results["raceId"] = 1133
    else:
        race_results["circuitRef"] = "hungaroring"
        race_results["raceId"] = 1134
    race_results = pd.merge(race_results, circuits, on="circuitRef")
    race_results = pd.merge(race_results, constructors, left_on="TeamId", right_on="constructorRef")
    race_results.drop(columns=["DriverId", "TeamName", "TeamId", "Points", "number", "code", "country", "lat", "lng", "alt", "url_y", "constructorRef", "name_y", "nationality_y", "url", "forename", "surname", "dob", "nationality_x", "url_x", "name_x", "location"], inplace=True)
    race_results.rename(columns={"Position":"position", "GridPosition":"grid"}, inplace=True)
    final = pd.concat([final, race_results], axis=0)

In [10]:
final.replace(to_replace="\\N", value=np.nan, inplace=True)

In [11]:
final.position = final.position.fillna(0).astype(np.int64, errors='ignore')
final.date = pd.to_datetime(final.date)
final["hour"] = final.time.str.replace(":.+", "", regex=True).astype(int)

In [12]:
final.to_csv("data/final.csv", index=False)